# 🧪 AiStock 动态价格系统 - 调试 Notebook (优化版)
> ✅ 修复语法错误 | ✅ 增强可视化 | ✅ 性能诊断 | ✅ 批量对比

## 🔧 1. 环境配置与路径注入

In [ ]:
import sys
import os
from pathlib import Path
import warnings
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import traceback

# 禁用警告
warnings.filterwarnings('ignore')

# 注入项目根目录（假设 notebook 放在 notebooks/ 下）
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"✅ 项目根目录: {PROJECT_ROOT}")
print(f"📁 工作目录: {os.getcwd()}")
print(f"🐍 Python 版本: {sys.version.split()[0]}")

## 📝 2. 日志配置与智能导入

In [ ]:
import logging
import importlib

# 配置彩色日志（终端支持时）
class ColoredFormatter(logging.Formatter):
    COLORS = {'DEBUG': '\033[36m', 'INFO': '\033[32m', 'WARNING': '\033[33m', 
              'ERROR': '\033[31m', 'CRITICAL': '\033[35m', 'RESET': '\033[0m'}
    
    def format(self, record):
        color = self.COLORS.get(record.levelname, self.COLORS['RESET'])
        record.levelname = f"{color}{record.levelname}{self.COLORS['RESET']}"
        return super().format(record)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("DebugMain")

def safe_import(import_path, fallback=None):
    """安全动态导入，支持模块.类名格式"""
    try:
        parts = import_path.split('.')
        if len(parts) == 1:
            return __import__(parts[0])
        module_path, obj_name = '.'.join(parts[:-1]), parts[-1]
        module = importlib.import_module(module_path)
        return getattr(module, obj_name)
    except Exception as e:
        logger.warning(f"⚠️ 导入 {import_path} 失败: {type(e).__name__}: {e}")
        return fallback

# 核心服务导入（按需启用）
ConfigService = safe_import('base_services.config_service.ConfigService')
CacheService = safe_import('base_services.cache_service.CacheService')
DatabaseReader = safe_import('data_services.database_reader.DatabaseReader')
TDXAdapter = safe_import('data_services.tdx_adapter.TDXAdapter')
AKAdapter = safe_import('data_services.ak_adapter.AKAdapter')
DataLoadingService = safe_import('data_services.data_loading_service.DataLoadingService')
DynamicPriceEngine = safe_import('dynamic_price_system.core.dynamic_price_engine.DynamicPriceEngine')

logger.info("🚀 调试环境初始化完成")

## ⚙️ 3. 服务初始化与配置加载

In [ ]:
# 1. 配置服务（优先 YAML，失败则使用 Mock）
CONFIG_YAML = PROJECT_ROOT / "config" / "dynamic_price" / "stocks_config.yaml"
if CONFIG_YAML.exists() and ConfigService:
    config = ConfigService(system_name='dynamic_price')
    logger.info(f"✅ 配置服务加载: {config.config.get('system.name', 'unknown')}")
else:
    # Mock 配置（调试用）
    class MockConfig:
        def __init__(self):
            self.config = {
                'system': {'name': 'AiStock Debug'},
                'stocks': [
                    {'code': '600938', 'name': '中国海油', 'sector': '油气开采', 'params': {
                        'stop_loss_atr_mult': 3.0, 'entry_ma_period': 20,
                        'target_multiplier': 1.20, 'min_fundamental_score': 60,
                        'macro_sensitivity': 1.7
                    }},
                    {'code': '601899', 'name': '紫金矿业', 'sector': '黄金', 'params': {
                        'stop_loss_atr_mult': 2.5, 'entry_ma_period': 20,
                        'target_multiplier': 1.25, 'min_fundamental_score': 65,
                        'macro_sensitivity': 1.9
                    }},
                ],
                'macro_indicators': {
                    'brent_crude': {'code': 'OIL', 'source': 'external'},
                    'comex_gold': {'code': 'GC', 'source': 'external'},
                    'pmi': {'code': '3_PMI', 'source': 'tdx'},
                },
                'cache': {'max_size': 2000, 'ttl': 300}
            }
        def get(self, key, default=None):
            keys = key.split('.')
            val = self.config
            for k in keys:
                if isinstance(val, dict):
                    val = val.get(k)
                else:
                    return default
            return val if val is not None else default
    config = MockConfig()
    logger.info("⚠️ 使用 Mock 配置服务（调试模式）")

# 2. 缓存服务
cache_config = config.config.get('cache', {})
cache = CacheService(
    max_size=cache_config.get('max_size', 2000), 
    ttl=cache_config.get('ttl', 300)
) if CacheService else None
logger.info(f"✅ 缓存服务: max_size={cache_config.get('max_size', 2000)}, ttl={cache_config.get('ttl', 300)}s")

# 3. 数据库服务（调试期降级为 Mock）
try:
    db_config = config.config.get('database', {})
    db_reader = DatabaseReader(
        db_config.get('DATABASE_ENGINES', {}), 
        db_config.get('DB_POOL_CONFIG', {})
    ) if DatabaseReader else None
    logger.info("✅ 数据库服务初始化" if db_reader else "⚠️ 数据库服务为 Mock")
except Exception as e:
    db_reader = None
    logger.warning(f"⚠️ 数据库初始化失败: {e}，使用 Mock")

# 4. TDX 适配器（调试期可跳过）
tdx_config = config.config.get('tdx', {})
tdx = TDXAdapter(tdx_config) if tdx_config.get('use_tdx') and TDXAdapter else None
if tdx:
    logger.info(f"✅ TDX 适配器: {tdx_config.get('exhq_host')}:{tdx_config.get('exhq_port')}")

# 5. 外部数据适配器 (AKShare)
ak = AKAdapter() if AKAdapter else None
if ak:
    logger.info("✅ AKShare 外部数据适配器就绪")

## 📊 4. 数据加载模块调试

In [ ]:
# 初始化数据加载服务（带 Mock 降级）
if DataLoadingService:
    data_loader = DataLoadingService(
        config_service=config,
        cache_service=cache,
        database_reader=db_reader,
        tdx_adapter=tdx,
        ak_adapter=ak,
        enable_cache=True
    )
    logger.info("✅ DataLoadingService 初始化完成")
else:
    # Mock DataLoadingService（调试用）
    class MockDataLoader:
        def load_all_macro_indicators(self, codes):
            return {
                'brent_crude': 94.21, 'comex_gold': 4859.97,
                'pmi': 51.2, 'm2_growth': 9.5, 'usd_cny': 7.22
            }
        def load_stock_daily(self, code, min_days=200):
            # 生成模拟行情数据（符合 TechnicalCalculator 输入规范）
            dates = pd.date_range(end=datetime.today(), periods=min_days, freq='D')
            np.random.seed(hash(code) % 2**32)
            close = np.cumprod(1 + np.random.normal(0.0005, 0.02, min_days)) * 40
            return pd.DataFrame({
                'datetime': dates,
                'open': close * (1 + np.random.randn(min_days) * 0.01),
                'high': np.maximum(close * (1 + np.abs(np.random.randn(min_days) * 0.015)), close),
                'low': np.minimum(close * (1 - np.abs(np.random.randn(min_days) * 0.015)), close),
                'close': close,
                'volume': np.random.randint(1_000_000, 8_000_000, min_days)
            })
        def load_stock_financials(self, code):
            return pd.DataFrame([{
                'revenue_growth': 18.5, 'profit_growth': 22.1, 'roe': 16.3,
                'gross_margin': 32.0, 'debt_ratio': 35.2
            }])
        def get_cache_stats(self):
            return {'hit_rate': 0.85, 'hits': 120, 'misses': 21, 'size': '141/2000', 'ttl': 300}
        def close(self): pass
    data_loader = MockDataLoader()
    logger.info("⚠️ 使用 Mock DataLoadingService（调试模式）")

# 测试宏观指标智能路由（带耗时统计）
macro_codes = ['brent_crude', 'comex_gold', 'lme_copper', 'pmi', 'm2_growth', 'usd_cny']
start = time.time()
macro_data = data_loader.load_all_macro_indicators(macro_codes)
macro_time = time.time() - start

print(f"\n📊 宏观指标加载 ({macro_time:.2f}s):")
for k, v in macro_data.items():
    status = "✅" if v is not None else "❌"
    print(f"  {status} {k}: {v if isinstance(v, (int, float)) else 'N/A'}")

# 缓存统计（如果启用）
if cache:
    stats = data_loader.get_cache_stats()
    print(f"\n📈 缓存统计: 命中率={stats['hit_rate']:.1%}, 大小={stats['size']}")

In [ ]:
# 股票数据加载测试（单标的）
test_code = '600938'  # 中国海油（调试用）
start = time.time()
test_df = data_loader.load_stock_daily(test_code, min_days=200)
load_time = time.time() - start

if test_df is not None and not test_df.empty:
    print(f"\n✅ 行情数据加载: {test_code} | {len(test_df)}条 | {load_time:.2f}s")
    print(test_df[['datetime', 'open', 'high', 'low', 'close', 'volume']].tail(3).to_string(index=False))
else:
    logger.warning(f"⚠️ 股票 {test_code} 数据加载失败或为空")

## 🎯 5. 动态价格引擎 - 单标的深度调试

In [ ]:
# 初始化计算引擎（带 Mock 降级）
if DynamicPriceEngine:
    engine = DynamicPriceEngine(config_service=config)
    logger.info("✅ DynamicPriceEngine 初始化完成")
else:
    # Mock Engine（调试用）
    class MockEngine:
        def calculate_single(self, code, sector, stock_data, financial_data, macro_data, stock_params):
            # 模拟计算结果（符合真实输出结构）
            entry = stock_data['close'].iloc[-1] * 0.95
            stop = entry * 0.92
            target = entry * 1.20
            return {
                'code': code, 'name': '中国海油', 'sector': sector,
                'calc_time': datetime.now().isoformat(),
                'prices': {'current': float(stock_data['close'].iloc[-1]), 'entry': round(entry,2), 'stop_loss': round(stop,2), 'target': round(target,2)},
                'factors': {'technical': 1.0, 'fundamental': 0.975, 'macro': 1.056, 'composite': 1.009},
                'scores': {'fundamental': 50.0, 'pl_ratio': round((target-entry)/(entry-stop),2)},
                'signals': {'trend': 'bullish', 'rsi_zone': 'neutral'},
                'recommendation': '观望',
                'params_used': stock_params
            }
        def calculate_batch(self, inputs):
            return [self.calculate_single(**inp) for inp in inputs]
    engine = MockEngine()
    logger.info("⚠️ 使用 Mock DynamicPriceEngine（调试模式）")

# 准备输入参数（从配置提取）
stock_cfg = next((s for s in config.get('stocks', []) if s.get('code') == test_code), {})
if not stock_cfg:
    logger.error(f"❌ 未找到标的配置: {test_code}")
else:
    stock_params = stock_cfg.get('params', {})
    
    # 财务数据加载（带异常处理）
    try:
        fin_df = data_loader.load_stock_financials(test_code)
        financial_data = fin_df.to_dict(orient='records')[0] if not fin_df.empty else {}
    except Exception as e:
        logger.warning(f"⚠️ 财务数据加载失败: {e}")
        financial_data = {}
    
    # 执行计算（带耗时统计 + 异常捕获）
    start = time.time()
    try:
        result = engine.calculate_single(
            code=test_code,
            name=stock_cfg.get('name', '未知'),
            sector=stock_cfg.get('sector', '未知'),
            stock_data=test_df,
            financial_data=financial_data,
            macro_data=macro_data,
            stock_params=stock_params
        )
        calc_time = time.time() - start
        
        if result:
            print(f"\n✅ 单标的计算成功! ({calc_time:.2f}s)")
            print("\n📋 计算结果摘要:")
            print(f"  • 当前价: ¥{result['prices']['current']:.2f}")
            print(f"  • 入场价: ¥{result['prices']['entry']:.2f} | 止损: ¥{result['prices']['stop_loss']:.2f} | 目标: ¥{result['prices']['target']:.2f}")
            print(f"  • 盈亏比: {result['scores']['pl_ratio']:.2f}x")
            print(f"  • 综合因子: {result['factors']['composite']:.3f} (基本面:{result['factors']['fundamental']:.3f} × 宏观:{result['factors']['macro']:.3f})")
            print(f"  • 建议: {result['recommendation']} | 趋势: {result['signals']['trend']}")
        else:
            logger.warning("❌ 计算返回 None，请检查日志或数据充分性")
    except Exception as e:
        logger.error(f"❌ 计算异常: {e}")
        traceback.print_exc()

## 🚀 6. 批量计算与性能诊断

In [ ]:
# 批量计算配置（取前 3 只标的调试）
batch_codes = [s.get('code') for s in config.get('stocks', [])[:3] if s.get('code')]
logger.info(f"🚀 开始批量计算: {len(batch_codes)} 只标的")

batch_inputs = []
for sc in config.get('stocks', []):
    code = sc.get('code')
    if code not in batch_codes:
        continue
    try:
        stock_data = data_loader.load_stock_daily(code, min_days=200)
        fin_df = data_loader.load_stock_financials(code)
        financial_data = fin_df.to_dict(orient='records')[0] if not fin_df.empty else {}
        
        batch_inputs.append({
            'code': code,
            'name': sc.get('name', '未知'),
            'sector': sc.get('sector', '未知'),
            'stock_data': stock_data,
            'financial_data': financial_data,
            'macro_data': macro_data,
            'params': sc.get('params', {})
        })
    except Exception as e:
        logger.warning(f"⚠️ 准备 {code} 输入失败: {e}")

# 执行批量计算（带性能统计）
start = time.time()
batch_results = engine.calculate_batch(batch_inputs) if hasattr(engine, 'calculate_batch') else []
batch_time = time.time() - start

print(f"\n✅ 批量计算完成: {len(batch_results)}/{len(batch_inputs)} 成功 | 耗时: {batch_time:.2f}s | 平均: {batch_time/max(1,len(batch_inputs)):.2f}s/只")

# 结果摘要表格（扁平化展示）
if batch_results:
    summary_data = []
    for r in batch_results:
        summary_data.append({
            '代码': r['code'],
            '名称': r.get('name', '未知'),
            '板块': r['sector'],
            '当前价': r['prices']['current'],
            '入场价': r['prices']['entry'],
            '止损价': r['prices']['stop_loss'],
            '目标价': r['prices']['target'],
            '盈亏比': r['scores']['pl_ratio'],
            '综合因子': r['factors']['composite'],
            '建议': r['recommendation']
        })
    
    df_summary = pd.DataFrame(summary_data)
    print("\n📊 批量结果摘要:")
    print(df_summary.to_string(index=False))

## 📈 7. 增强可视化模块（交互式 Plotly）

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# 7.1 单标的价格区间可视化（增强版）
def plot_price_analysis(result):
    """增强版价格区间可视化"""
    if not result or 'prices' not in result:
        return None
    
    p = result['prices']
    f = result['factors']
    
    # 动态计算 Y 轴范围（避免硬编码截断数据）
    all_p = [v for v in p.values() if isinstance(v, (int, float))]
    y_min, y_max = min(all_p), max(all_p)
    padding = (y_max - y_min) * 0.3
    
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[f"价格区间分析 ({result['code']}:{result.get('name','')})", "因子分解"],
        vertical_spacing=0.15,
        row_heights=[0.7, 0.3]
    )
    
    # 第一行：价格区间（修复 customdata 维度问题，改用 hovertext）
    # 当前价（蓝色星形）
    fig.add_trace(go.Scatter(
        x=['当前价'], y=[p['current']],
        mode='markers',
        marker=dict(size=16, color='blue', symbol='star', line=dict(width=2, color='white')),
        name='当前价',
        hovertemplate='<b>当前价</b><br>¥%{y:.2f}<extra></extra>'
    ), row=1, col=1)
    
    # 入场区间（绿色虚线 + 悬停提示）
    entry_low, entry_high = p['entry'] * 0.98, p['entry'] * 1.02
    fig.add_trace(go.Scatter(
        x=['入场区间', '入场区间'], y=[entry_low, entry_high],
        mode='lines',
        line=dict(color='green', width=4, dash='dash'),
        name='入场区间',
        hovertemplate='<b>入场区间</b><br>¥' + f'{entry_low:.2f} - {entry_high:.2f}<extra></extra>'
    ), row=1, col=1)
    
    # 止损价（红色叉号）
    fig.add_trace(go.Scatter(
        x=['止损价'], y=[p['stop_loss']],
        mode='markers+text',
        marker=dict(size=14, color='red', symbol='x', line=dict(width=2)),
        text=['止损'], textposition='bottom center',
        name='止损价',
        hovertemplate='<b>止损价</b><br>¥%{y:.2f}<extra></extra>'
    ), row=1, col=1)
    
    # 目标价（蓝色菱形 + 盈亏比标注）
    target_text = f"目标 (盈亏比:{result['scores']['pl_ratio']:.1f}x)"
    fig.add_trace(go.Scatter(
        x=['目标价'], y=[p['target']],
        mode='markers+text',
        marker=dict(size=14, color='darkblue', symbol='diamond', line=dict(width=2)),
        text=[target_text], textposition='top center',
        name='目标价',
        hovertemplate=f"<b>目标价</b><br>¥{p['target']:.2f}<br>盈亏比：{result['scores']['pl_ratio']:.1f}x<extra></extra>"
    ), row=1, col=1)
    
    # 潜在盈利区间（浅绿背景色带，使用 add_shape 替代 fill='toself'）
    fig.add_shape(
        type="rect", xref="x", yref="y",
        x0=0.6, x1=1.4,  # 分类轴索引：0=当前价, 1=入场区间。0.6~1.4 覆盖该类别
        y0=entry_low, y1=p['target'],
        fillcolor="rgba(0, 128, 0, 0.1)", line_width=0, layer="below"
    )
    
    # 第二行：因子分解（条形图）
    factors = [
        ('技术面', 1.0),
        ('基本面', f['fundamental']),
        ('宏观面', f['macro']),
        ('综合', f['composite'])
    ]
    factor_names, factor_values = zip(*factors)
    
    colors = ['gray', 'orange', 'purple', 'darkblue']
    fig.add_trace(go.Bar(
        x=factor_names, y=factor_values,
        marker_color=colors,
        name='调整因子',
        text=[f"{v:.3f}" for v in factor_values],
        textposition='auto',
        hovertemplate='<b>%{x}</b><br>因子：%{y:.3f}<extra></extra>'
    ), row=2, col=1)
    
    # 更新布局（修复 f-string 引号冲突，外层双引号）
    fig.update_layout(
        height=600,
        hovermode='closest',
        xaxis=dict(title='价格类型', showgrid=False, categoryorder='array', categoryarray=['当前价', '入场区间', '止损价', '目标价']),
        yaxis=dict(title='价格 (元)', range=[y_min - padding, y_max + padding]),
        yaxis2=dict(title='调整因子', range=[0.85, 1.15]),
        legend=dict(orientation='h', yanchor='bottom', y=-0.1, xanchor='center', x=0.5),
        showlegend=True
    )
    
    # 添加注释（修复引号问题）
    fig.add_annotation(
        x=0.5, y=1.02,
        text=f"板块:{result['sector']} | 趋势:{result['signals']['trend']} | RSI:{result['signals']['rsi_zone']}",
        showarrow=False, bgcolor='lightgray', font=dict(size=10),
        xref='paper', yref='paper'
    )
    
    return fig

# 7.2 批量结果对比可视化（散点图）
def plot_batch_comparison(results):
    """批量结果对比图"""
    if not results:
        return None
    
    # 准备数据（确保原生类型）
    df = pd.DataFrame([{
        '代码': r['code'],
        '名称': r.get('name', '未知'),
        '板块': r['sector'],
        '盈亏比': float(r['scores']['pl_ratio']),
        '综合因子': float(r['factors']['composite']),
        '建议': r['recommendation'],
        '入场价': float(r['prices']['entry']),
        '目标价': float(r['prices']['target'])
    } for r in results])
    
    # 建议颜色映射（修复颜色键匹配）
    color_map = {'强烈推荐': 'green', '推荐': 'blue', '观望': 'orange', '谨慎': 'red'}
    
    # 盈亏比散点图（交互式）
    fig = px.scatter(
        df, x='综合因子', y='盈亏比',
        color='建议', color_discrete_map=color_map,
        size='目标价', hover_name='名称',
        title='🎯 批量标的对比（综合因子 × 盈亏比）',
        labels={'综合因子': '综合调整因子', '盈亏比': '盈亏比 (x)'},
        size_max=30, opacity=0.8
    )
    
    # 添加参考线（中性因子 + 盈亏比阈值）
    fig.add_vline(x=1.0, line_dash='dash', line_color='gray', annotation_text='中性因子')
    fig.add_hline(y=2.0, line_dash='dash', line_color='blue', annotation_text='盈亏比阈值')
    
    fig.update_layout(height=500, hovermode='closest')
    return fig

# 执行可视化（带异常处理）
print("\n🎨 生成可视化图表...")

# 单标的详细分析（如果存在结果）
if result and 'prices' in result:
    try:
        fig1 = plot_price_analysis(result)
        if fig1:
            fig1.show()
    except Exception as e:
        logger.error(f"❌ 单标的可视化失败: {e}")

# 批量对比（如果存在批量结果）
if batch_results:
    try:
        fig2 = plot_batch_comparison(batch_results)
        if fig2:
            fig2.show()
    except Exception as e:
        logger.error(f"❌ 批量可视化失败: {e}")

## 📊 8. 性能诊断与缓存分析

In [ ]:
# 性能诊断面板（如果缓存服务可用）
if cache and hasattr(data_loader, 'get_cache_stats'):
    try:
        stats = data_loader.get_cache_stats()
        
        # 创建性能仪表板（多子图布局）
        fig_perf = make_subplots(
            rows=1, cols=3,
            subplot_titles=['缓存命中率', '请求分布', '容量使用'],
            specs=[[{'type': 'indicator'}, {'type': 'pie'}, {'type': 'indicator'}]]
        )
        
        # 命中率仪表（带阈值参考）
        fig_perf.add_trace(go.Indicator(
            mode='gauge+number+delta',
            value=stats['hit_rate'],
            domain={'x': [0, 0.3], 'y': [0, 1]},
            title={'text': '命中率'},
            delta={'reference': 0.8},
            gauge={'axis': {'range': [0, 1]}, 'bar': {'color': 'darkblue'}}
        ), row=1, col=1)
        
        # 请求分布饼图（命中/未命中）
        fig_perf.add_trace(go.Pie(
            labels=['命中', '未命中'],
            values=[stats['hits'], stats['misses']],
            marker_colors=['green', 'orange'],
            hole=0.4,
            hovertemplate='<b>%{label}</b><br>次数：%{value}<extra></extra>'
        ), row=1, col=2)
        
        # 容量使用进度条（当前/最大）
        current, max_size = map(int, stats['size'].split('/'))
        fig_perf.add_trace(go.Indicator(
            mode='gauge+number',
            value=current / max_size,
            domain={'x': [0.7, 1], 'y': [0, 1]},
            title={'text': '容量'},
            gauge={'axis': {'range': [0, 1]}, 'bar': {'color': 'purple'}}
        ), row=1, col=3)
        
        fig_perf.update_layout(
            title='📈 性能诊断面板',
            height=300,
            margin=dict(l=20, r=20, t=50, b=20)
        )
        fig_perf.show()
    except Exception as e:
        logger.warning(f"⚠️ 性能诊断可视化失败: {e}")

## 🧹 9. 资源清理与调试总结

In [ ]:
# 清理资源（带异常处理）
try:
    if hasattr(data_loader, 'close'):
        data_loader.close()
    if cache and hasattr(cache, 'clear'):
        cache.clear()
    if tdx and hasattr(tdx, 'close'):
        tdx.close()
    logger.info("✅ 资源清理完成")
except Exception as e:
    logger.warning(f"⚠️ 资源清理警告: {e}")

# 最终总结报告（关键指标回顾）
print("\n" + "="*70)
print("🏁 调试流水线执行完毕")
print("="*70)
print("💡 关键指标回顾:")
if batch_results:
    avg_pl = np.mean([r['scores']['pl_ratio'] for r in batch_results])
    avg_factor = np.mean([r['factors']['composite'] for r in batch_results])
    rec_counts = pd.Series([r['recommendation'] for r in batch_results]).value_counts()
    print(f"  • 平均盈亏比: {avg_pl:.2f}x")
    print(f"  • 平均综合因子: {avg_factor:.3f}")
    print(f"  • 建议分布: {dict(rec_counts)}")
print("🔍 后续建议:")
print("  1. 检查日志中 ⚠️ 警告，确认降级策略是否合理")
print("  2. 验证 macro_data 路由：外部源 → TDX → DB 降级链")
print("  3. 监控缓存命中率：>80% 为优，<50% 需优化加载策略")
print("  4. 生产环境：关闭 DEBUG 日志，启用性能监控")
print("="*70)